# APTOS 2019 — exp300_d1_dropout_cosine submission

Self-contained Kaggle inference notebook for the `exp300_d1_dropout_cosine` checkpoint.
Mirrors `src/models.py` (ResNet50 + GeM + Dropout(0.3)+Linear head) and `src/dataset.py`
(Ben-Graham bbox crop + local contrast + ImageNet norm) exactly.

**Before running:** upload the checkpoint folder as a Kaggle dataset, then edit
`CHECKPOINT_DATASET_DIR` in the Configuration cell to match its `/kaggle/input/<slug>` path.


In [ ]:
# Cell 1 — imports & device
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}, Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# Cell 2 — configuration
NUM_CLASSES = 5
IMAGE_SIZE  = 512
BATCH_SIZE  = 32
NUM_WORKERS = 2

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# === EDIT THESE FOR YOUR KAGGLE DATASET ===
# Upload the contents of checkpoints/exp300_d1_dropout_cosine/ as a Kaggle dataset,
# then set CHECKPOINT_DATASET_DIR to its /kaggle/input/<slug> path.
CHECKPOINT_DATASET_DIR = '/kaggle/input/exp300-d1-dropout-cosine'
CHECKPOINT_NAME        = 'exp300_d1_dropout_cosine_best.pth'  # or '..._last.pth'
CHECKPOINT_PATH        = f'{CHECKPOINT_DATASET_DIR}/{CHECKPOINT_NAME}'

TEST_CSV        = '/kaggle/input/aptos2019-blindness-detection/test.csv'
TEST_IMAGES_DIR = '/kaggle/input/aptos2019-blindness-detection/test_images'
OUTPUT_PATH     = '/kaggle/working/submission.csv'

# Must match exp300 training config (src/config.py exp_id=300)
HEAD_DROPOUT = 0.3
GEM_P        = 3.0


In [ ]:
# Cell 3 — model: ResNet50 + GeM + Dropout(0.3)+Linear head (mirrors src/models.py)
class GeM(nn.Module):
    def __init__(self, p: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(
            x.clamp(min=self.eps).pow(self.p),
            (x.size(-2), x.size(-1)),
        ).pow(1.0 / self.p)


def build_exp300_model():
    # weights=None — our checkpoint will overwrite, and Kaggle's torchvision has
    # known issues loading ImageNet weights (see notebooks/kaggle_runner.ipynb).
    model = tvm.resnet50(weights=None)
    model.avgpool = GeM(p=GEM_P)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=HEAD_DROPOUT),
        nn.Linear(in_features, NUM_CLASSES),
    )
    return model


In [ ]:
# Cell 4 — Ben-Graham preprocessing (verbatim from src/dataset.py: bbox crop, no circular mask)
def ben_graham_preprocess(image: np.ndarray, size: int = IMAGE_SIZE) -> np.ndarray:
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest)
        image = image[y:y + h, x:x + w]
    image = cv2.resize(image, (size, size), interpolation=cv2.INTER_LINEAR)
    gauss = cv2.GaussianBlur(image, (0, 0), sigmaX=size / 30)
    image = cv2.addWeighted(image, 4, gauss, -4, 128)
    return image


In [ ]:
# Cell 5 — test dataset (mirrors DRDataset image-load + normalize sequence)
class APTOSTestDataset(Dataset):
    def __init__(self, csv_path, img_dir, size=IMAGE_SIZE):
        self.df = pd.read_csv(csv_path)         # 'id_code' column
        self.img_dir = img_dir
        self.size = size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        code = self.df.iloc[idx]['id_code']
        image = cv2.imread(os.path.join(self.img_dir, f'{code}.png'))   # BGR
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = ben_graham_preprocess(image, self.size)
        image_t = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
        for c in range(3):
            image_t[c] = (image_t[c] - IMAGENET_MEAN[c]) / IMAGENET_STD[c]
        return image_t, code


In [ ]:
# Cell 6 — load checkpoint (raw state_dict from torch.save(model.state_dict(), ...))
model = build_exp300_model().to(DEVICE)
state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
if isinstance(state_dict, dict) and 'model_state_dict' in state_dict:
    state_dict = state_dict['model_state_dict']
result = model.load_state_dict(state_dict, strict=True)
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'Loaded {CHECKPOINT_NAME}  |  params: {n_params/1e6:.2f}M')
print(f'load_state_dict: {result}')


In [ ]:
# Cell 7 — DataLoader
test_ds = APTOSTestDataset(TEST_CSV, TEST_IMAGES_DIR)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
print(f'Test images: {len(test_ds)}')


In [ ]:
# Cell 8 — inference (no TTA — matches exp300 training config)
all_ids, all_preds = [], []
with torch.no_grad():
    for images, codes in tqdm(test_loader, desc='Inference'):
        images = images.to(DEVICE, non_blocking=True)
        logits = model(images)                    # [B, 5]
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_ids.extend(codes)

dist = pd.Series(all_preds).value_counts().sort_index().to_dict()
print(f'Total predictions: {len(all_preds)}')
print(f'Predicted grade distribution: {dist}')


In [ ]:
# Cell 9 — write submission
submission = pd.DataFrame({'id_code': all_ids, 'diagnosis': all_preds})
submission.to_csv(OUTPUT_PATH, index=False)
print(f'Wrote {OUTPUT_PATH}  shape={submission.shape}')
submission.head()
